# Limpeza dos mapas de calor (`heatmaps/`)

Uma imagem PNG por assinante e data (31/07 e 30/09/2026), com as rotas dos treinos ao ar livre nos 90 dias até a data.
As imagens brutas não são alteradas nem movidas: a limpeza é um **índice** das imagens, com marcas de qualidade, para a
etapa de extração (contar pixels ou ler com modelo de imagem) partir de uma base conferida.

| Tabela | Uma linha por | Para quê |
|---|---|---|
| `heatmaps_indice.parquet` | imagem (conta × data) | caminho, formato, pixels de rota e marcas de qualidade |
| `heatmaps_conta.parquet` | conta com imagem em alguma data | as duas datas lado a lado, no formato do modelo |

**Cores.** O dicionário descreve só o fundo (verde parque, azul água, areia praia, amarelo avenida). O levantamento de todas
as imagens mostra 6 cores exatas de fundo, presentes em todas, e milhares de tons quentes (amarelo forte → laranja) que são a
escala de calor da rota. Pixel de rota = qualquer pixel fora das 6 cores de fundo.

Variáveis mais ricas (quanto da rota passa em parque, água, praia ou avenida; intensidade) ficam para a extração.
O hash dos 25.478 arquivos da pasta é conferido no início e no fim.

In [1]:
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BRUTOS = ROOT / "dados" / "brutos"
HEAT = BRUTOS / "heatmaps"
PROCESSADOS = ROOT / "dados" / "processados"
SAIDAS = ROOT / "saidas"
DATAS = {"2026-07-31": 12_426, "2026-09-30": 13_047}   # contagens do cabeçalho da seção 14 do dicionário
JANELA = pd.Timedelta(days=90)
AR_LIVRE = ["corrida", "caminhada", "pedal"]
FUNDO = {
    (236, 233, 226): "terreno",
    (252, 252, 250): "rua",
    (197, 226, 175): "parque",
    (170, 207, 230): "agua",
    (250, 236, 190): "avenida",
    (240, 228, 196): "praia",
}

todos = sorted(p for p in HEAT.rglob("*") if p.is_file())

def md5(path):
    return hashlib.md5(path.read_bytes()).hexdigest()

hash_inicio = {p.relative_to(HEAT).as_posix(): md5(p) for p in todos}
print(f"arquivos na pasta: {len(todos):,}")

registro = []

def registrar(tabela, passo, n, obs=""):
    registro.append({"tabela": tabela, "passo": passo, "linhas": n, "observacao": obs})
    print(f"[{tabela}] {passo}: {n:,} linhas" + (f" ({obs})" if obs else ""))

arquivos na pasta: 25,478


## 0. Inventário dos arquivos

O zip veio com uma pasta `__MACOSX` (metadados do Finder, arquivos `._*.png` que não são imagens).
Cada data tem a pasta `<data>/` dentro de `heatmaps_<data>/`.

In [2]:
inv = pd.DataFrame({"caminho": [p.relative_to(HEAT).as_posix() for p in todos]})
inv["macosx"] = inv["caminho"].str.contains("__MACOSX")
inv["nome"] = inv["caminho"].str.split("/").str[-1]
display(inv.groupby("macosx").agg(arquivos=("caminho", "size"), exemplo=("caminho", "first")))

imagens = inv.loc[~inv["macosx"]].copy()
imagens["data"] = imagens["caminho"].str.split("/").str[-2]
imagens["user_id"] = imagens["nome"].str.removesuffix(".png")
assert imagens["nome"].str.fullmatch(r"u_[0-9a-f]{8}\.png").all()
assert imagens["caminho"].str.fullmatch(r"heatmaps_(\d{4}-\d{2}-\d{2})/\1/u_[0-9a-f]{8}\.png").all()
contagem = imagens.groupby("data").size()
display(pd.DataFrame({"imagens": contagem, "dicionário": pd.Series(DATAS)}))
assert contagem.to_dict() == DATAS
assert not imagens.duplicated(["data", "user_id"]).any()
registrar("indice", "0. Imagens (sem __MACOSX)", len(imagens), f"{inv['macosx'].sum()} arquivos de __MACOSX ignorados")

,arquivos,exemplo
macosx,,
False,25473,heatmaps_2026-07-31/2026-07-31/u_0001651d.png
True,5,heatmaps_2026-07-31/__MACOSX/2026-07-31/._u_00...


,imagens,dicionário
2026-07-31,12426,12426
2026-09-30,13047,13047


[indice] 0. Imagens (sem __MACOSX): 25,473 linhas (5 arquivos de __MACOSX ignorados)


## 1. Formato e leitura de cada imagem

Para cada imagem: tamanho, dimensões, profundidade da paleta, hash e a contagem de pixels por cor.

In [3]:
linhas = []
for r in imagens.itertuples():
    p = HEAT / r.caminho
    b = p.read_bytes()
    im = Image.open(p)
    im.load()
    cores = im.convert("RGB").getcolors(maxcolors=192 * 192)
    fundo = {nome: 0 for nome in FUNDO.values()}
    px_rota, cores_rota = 0, 0
    for k, c in cores:
        if c in FUNDO:
            fundo[FUNDO[c]] += k
        else:
            px_rota += k
            cores_rota += 1
    linhas.append({"caminho": r.caminho, "bytes": len(b), "md5": hashlib.md5(b).hexdigest(), "formato": im.format,
                   "modo": im.mode, "largura": im.width, "altura": im.height, "cores_imagem": len(cores),
                   "px_rota": px_rota, "cores_rota": cores_rota, **{f"px_{k}": v for k, v in fundo.items()}})
meta = pd.DataFrame(linhas)
idx = imagens.merge(meta, on="caminho", validate="one_to_one")

display(idx.groupby(["formato", "modo", "largura", "altura"]).size().rename("imagens").to_frame())
assert (idx["formato"] == "PNG").all() and (idx["largura"] == 192).all() and (idx["altura"] == 192).all()
px_total = idx[["px_rota"] + [f"px_{k}" for k in FUNDO.values()]].sum(axis=1)
assert (px_total == 192 * 192).all()
print(f"bytes: {idx['bytes'].min()} a {idx['bytes'].max()}; cores por imagem: {idx['cores_imagem'].min()} a {idx['cores_imagem'].max()}")

,,,,imagens
formato,modo,largura,altura,
PNG,P,192,192,25473


bytes: 601 a 9122; cores por imagem: 3 a 48


### 1.1 As cores de fundo

As 6 cores de fundo aparecem em (quase) todas as imagens; os tons da rota são quentes (vermelho ≥ azul).

In [4]:
presenca = pd.Series({k: (idx[f"px_{k}"] > 0).mean() for k in FUNDO.values()}, name="% das imagens com a cor").round(3)
display(presenca.to_frame())

tons = {}
for r in idx.sample(2000, random_state=0).itertuples():
    for k, c in Image.open(HEAT / r.caminho).convert("RGB").getcolors(maxcolors=192 * 192):
        if c not in FUNDO:
            tons[c] = tons.get(c, 0) + k
t = pd.DataFrame([(c[0], c[1], c[2], k) for c, k in tons.items()], columns=["r", "g", "b", "px"])
print(f"amostra de 2.000 imagens: {len(t):,} tons de rota; pixels de rota com vermelho ≥ azul: {t.loc[t['r'] >= t['b'], 'px'].sum() / t['px'].sum():.4%}")

,% das imagens com a cor
terreno,1.000
rua,1.000
parque,0.999
agua,0.704
avenida,0.999
praia,0.277


amostra de 2.000 imagens: 3,393 tons de rota; pixels de rota com vermelho ≥ azul: 99.9990%


## 2. Marcas de qualidade

- `mapa_sem_rota`: nenhum pixel fora do fundo.
- `igual_a_outra_data`: o mesmo arquivo, byte a byte, nas duas datas.
- `imagem_igual_a_outra_conta`: o mesmo arquivo em contas diferentes.

In [5]:
idx["fracao_rota"] = (idx["px_rota"] / (192 * 192)).round(5)
idx["mapa_sem_rota"] = idx["px_rota"] == 0
dup_conta = idx.groupby("md5")["user_id"].transform("nunique") > 1
idx["imagem_igual_a_outra_conta"] = dup_conta
idx["igual_a_outra_data"] = idx.duplicated(["user_id", "md5"], keep=False)

marcas = pd.DataFrame({
    "mapa_sem_rota": idx.groupby("data")["mapa_sem_rota"].sum(),
    "igual_a_outra_data": idx.groupby("data")["igual_a_outra_data"].sum(),
    "imagem_igual_a_outra_conta": idx.groupby("data")["imagem_igual_a_outra_conta"].sum(),
})
display(marcas)
print("contas com mapa sem rota:", sorted(idx.loc[idx["mapa_sem_rota"], "user_id"].unique().tolist()))
print("iguais nas duas datas também sem rota:", idx.loc[idx["igual_a_outra_data"], "mapa_sem_rota"].all())
assert idx["imagem_igual_a_outra_conta"].sum() == 0
display(idx.groupby("data")["px_rota"].describe().round(0))

,mapa_sem_rota,igual_a_outra_data,imagem_igual_a_outra_conta
data,,,
2026-07-31,5,2,0
2026-09-30,6,2,0


contas com mapa sem rota: ['u_0c2a9cf1', 'u_20d497f0', 'u_5a1e4a56', 'u_7936fa46', 'u_89ecca86', 'u_8d31200f', 'u_a9a2ad09', 'u_be043ed4', 'u_fe3ecfcf']
iguais nas duas datas também sem rota: True


,count,mean,std,min,25%,50%,75%,max
data,,,,,,,,
2026-07-31,12426.0,1382.0,1076.0,0.0,571.0,1043.0,1920.0,8396.0
2026-09-30,13047.0,1382.0,1080.0,0.0,566.0,1041.0,1930.0,8798.0


## 3. Cruzamento com assinaturas e treinos

O dicionário diz que só existe imagem para quem era assinante pagante na data **e** treinou ao ar livre nos 90 dias.
A primeira parte se testa com `assinaturas_limpa`; a segunda, com os treinos de corrida, caminhada e pedal de
`atividades_limpa`, na janela de 90 dias que termina no fim do dia da pasta.

In [6]:
assin = pd.read_parquet(PROCESSADOS / "assinaturas_limpa.parquet")
usuarios = pd.read_parquet(PROCESSADOS / "usuarios_limpa.parquet", columns=["user_id"])
at = pd.read_parquet(PROCESSADOS / "atividades_limpa.parquet", columns=["user_id", "inicio", "tipo"])
ar = at.loc[at["tipo"].isin(AR_LIVRE)]

assert idx["user_id"].isin(usuarios["user_id"]).all()
resumo = []
partes = []
for data in DATAS:
    d = pd.Timestamp(data)
    fim = d + pd.Timedelta(days=1)
    ativos = set(assin.loc[(assin["inicio"] <= d) & (assin["data_fim"].isna() | (assin["data_fim"] > d)), "user_id"])
    janela = ar.loc[(ar["inicio"] >= fim - JANELA) & (ar["inicio"] < fim)]
    n_ar = janela.groupby("user_id").size()
    x = idx.loc[idx["data"] == data, ["data", "user_id"]].copy()
    x["assinante_na_data"] = x["user_id"].isin(ativos)
    x["treinos_ar_livre_90d"] = x["user_id"].map(n_ar).fillna(0).astype("int64")
    partes.append(x)
    com_img = set(x["user_id"])
    treinou = set(n_ar.index) & ativos
    sem_img = treinou - com_img
    resumo.append({
        "data": data, "imagens": len(x), "assinantes na data": len(ativos),
        "imagem de quem não era assinante": int((~x["assinante_na_data"]).sum()),
        "assinantes que treinaram ao ar livre": len(treinou),
        "imagem sem treino ao ar livre na janela": int((x["treinos_ar_livre_90d"] == 0).sum()),
        "treinou ao ar livre e sem imagem": len(sem_img),
        "mediana de treinos: com imagem": float(n_ar.reindex(list(com_img)).median()),
        "mediana de treinos: sem imagem": float(n_ar.reindex(list(sem_img)).median()),
    })
idx = idx.merge(pd.concat(partes), on=["data", "user_id"], validate="one_to_one")
tab = pd.DataFrame(resumo).set_index("data").T
display(tab)
assert idx["assinante_na_data"].all()

data,2026-07-31,2026-09-30
imagens,12426.0,13047.0
assinantes na data,14032.0,14753.0
imagem de quem não era assinante,0.0,0.0
assinantes que treinaram ao ar livre,12643.0,13308.0
imagem sem treino ao ar livre na janela,316.0,349.0
treinou ao ar livre e sem imagem,533.0,610.0
mediana de treinos: com imagem,10.0,10.0
mediana de treinos: sem imagem,1.0,1.0


Hipótese testada: as imagens de quem não treinou ao ar livre na janela seriam os mapas sem rota. Não são.

In [7]:
display(pd.crosstab(idx["mapa_sem_rota"], idx["treinos_ar_livre_90d"] == 0, rownames=["mapa sem rota"],
                    colnames=["sem treino ao ar livre na janela"]))
sem_treino = idx.loc[idx["treinos_ar_livre_90d"] == 0]
print(f"imagens sem treino ao ar livre na janela: {len(sem_treino)}, com mediana de {sem_treino['px_rota'].median():.0f} pixels de rota "
      f"(todas as imagens: {idx['px_rota'].median():.0f})")
for dias in [89, 91]:
    n = 0
    for data in DATAS:
        fim = pd.Timestamp(data) + pd.Timedelta(days=1)
        jan = set(ar.loc[(ar["inicio"] >= fim - pd.Timedelta(days=dias)) & (ar["inicio"] < fim), "user_id"])
        n += (~idx.loc[idx["data"] == data, "user_id"].isin(jan)).sum()
    print(f"com janela de {dias} dias: {n} imagens sem treino ao ar livre (90 dias: {len(sem_treino)})")

sem treino ao ar livre na janela,False,True
mapa sem rota,,
False,24797,665
True,11,0


imagens sem treino ao ar livre na janela: 665, com mediana de 397 pixels de rota (todas as imagens: 1041)
com janela de 89 dias: 686 imagens sem treino ao ar livre (90 dias: 665)
com janela de 91 dias: 652 imagens sem treino ao ar livre (90 dias: 665)


A divergência acontece nos dois sentidos, nas duas datas, e não some mudando a janela em um dia. Quem treinou ao ar livre e
ficou sem imagem tem poucos treinos (mediana 1, contra 10 de quem tem imagem); as imagens sem treino na janela têm rota normal.
A presença da imagem não é um indicador exato de "treinou ao ar livre": a ausência de imagem fica como não informado.

## 4. Tabelas

In [8]:
COLS = ["data", "user_id", "caminho", "bytes", "md5", "largura", "altura", "cores_imagem", "px_rota", "fracao_rota",
        "cores_rota", "px_terreno", "px_rua", "px_parque", "px_agua", "px_avenida", "px_praia",
        "mapa_sem_rota", "igual_a_outra_data", "treinos_ar_livre_90d", "assinante_na_data"]
indice = idx[COLS].sort_values(["data", "user_id"]).reset_index(drop=True)
indice["data"] = pd.to_datetime(indice["data"])
indice["caminho"] = "dados/brutos/heatmaps/" + indice["caminho"]
registrar("indice", "4. heatmaps_indice", len(indice),
          f"{indice['mapa_sem_rota'].sum()} sem rota; {indice['igual_a_outra_data'].sum()} iguais nas duas datas")

larga = indice.assign(d=indice["data"].dt.strftime("%m%d")).pivot(index="user_id", columns="d",
                                                                    values=["px_rota", "mapa_sem_rota", "treinos_ar_livre_90d"])
larga.columns = [f"{a}_{b}" for a, b in larga.columns]
conta = larga.reset_index()
conta["tem_imagem_0731"] = conta["px_rota_0731"].notna()
conta["tem_imagem_0930"] = conta["px_rota_0930"].notna()
# mapa sem rota: rota desconhecida, não zero
for d in ["0731", "0930"]:
    conta[f"mapa_sem_rota_{d}"] = conta[f"mapa_sem_rota_{d}"].astype("boolean")
    conta[f"px_rota_{d}"] = conta[f"px_rota_{d}"].astype("Int64").where(~conta[f"mapa_sem_rota_{d}"].fillna(False))
    conta[f"treinos_ar_livre_90d_{d}"] = conta[f"treinos_ar_livre_90d_{d}"].astype("Int64")
conta["variacao_px_rota"] = (conta["px_rota_0930"] - conta["px_rota_0731"]).astype("Int64")
conta = conta[["user_id", "tem_imagem_0731", "tem_imagem_0930", "px_rota_0731", "px_rota_0930", "variacao_px_rota",
               "mapa_sem_rota_0731", "mapa_sem_rota_0930", "treinos_ar_livre_90d_0731", "treinos_ar_livre_90d_0930"]]
assert conta["user_id"].is_unique and conta["tem_imagem_0731"].sum() == DATAS["2026-07-31"] and conta["tem_imagem_0930"].sum() == DATAS["2026-09-30"]
display(pd.crosstab(conta["tem_imagem_0731"], conta["tem_imagem_0930"], margins=True, margins_name="total"))
registrar("conta", "4. heatmaps_conta", len(conta),
          f"{(conta['tem_imagem_0731'] & conta['tem_imagem_0930']).sum():,} com imagem nas duas datas")

[indice] 4. heatmaps_indice: 25,473 linhas (11 sem rota; 4 iguais nas duas datas)


tem_imagem_0930,False,True,total
tem_imagem_0731,,,
False,0,2501,2501
True,1880,10546,12426
total,1880,13047,14927


[conta] 4. heatmaps_conta: 14,927 linhas (10,546 com imagem nas duas datas)


## 5. Saídas e integridade dos brutos

In [9]:
for nome, df in {"heatmaps_indice.parquet": indice, "heatmaps_conta.parquet": conta}.items():
    df.to_parquet(PROCESSADOS / nome, index=False)
    pd.testing.assert_frame_equal(pd.read_parquet(PROCESSADOS / nome), df)
    print(f"dados/processados/{nome}: {len(df):,} linhas x {df.shape[1]} colunas")

pd.DataFrame(registro).to_csv(SAIDAS / "limpeza_heatmaps_registro.csv", index=False)
display(pd.DataFrame(registro))

hash_fim = {p.relative_to(HEAT).as_posix(): md5(p) for p in sorted(p for p in HEAT.rglob("*") if p.is_file())}
assert hash_fim == hash_inicio
print(f"dados/brutos/heatmaps/ inalterado ({len(hash_fim):,} arquivos):", hash_fim == hash_inicio)

dados/processados/heatmaps_indice.parquet: 25,473 linhas x 21 colunas
dados/processados/heatmaps_conta.parquet: 14,927 linhas x 10 colunas


,tabela,passo,linhas,observacao
0,indice,0. Imagens (sem __MACOSX),25473,5 arquivos de __MACOSX ignorados
1,indice,4. heatmaps_indice,25473,11 sem rota; 4 iguais nas duas datas
2,conta,4. heatmaps_conta,14927,"10,546 com imagem nas duas datas"


dados/brutos/heatmaps/ inalterado (25,478 arquivos): True
